# Fill missing query facets via zero-shot NLI classification

Imputes the missing values in the **`complexity`**, **`technicality`**, and **`sensitivity`**
facet columns of the scaleup query registry by reading each `query_text` with the
**same DeBERTa NLI checkpoint used in the attribution pipeline**
(`MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli`; cf. `thesis_config.NLI_MODEL`).

**Method (written for the methods chapter):**
- **Fill blanks only** — existing GEO-Bench labels are never overwritten.
- Zero-shot classification = NLI entailment of one hypothesis per candidate label
  (premise = the query, hypothesis = e.g. *"This is a simple factual question."*).
  This is the *same entailment mechanism* as the pipeline's AIS step, applied to queries.
- A **consistency check** re-classifies the rows that already had a label and reports
  agreement with those human labels — the validation number to quote in the write-up.
- Every imputed value is logged with its confidence score for an auditable record.

Outputs are **non-destructive**: a new `*_filled.parquet` is written and the original is untouched.

> Requirements: `pip install transformers torch pandas tqdm pyarrow`
>
> The three facets are scalar string columns; **31 + 29 + 51 = 111** cells are missing
> (the classifier detects missingness dynamically, so this is computed, not hard-coded).

## 1. Imports, device, and load the query registry

In [1]:
import pandas as pd
import torch
from pathlib import Path
from tqdm.notebook import tqdm
import transformers

print("transformers", transformers.__version__, "| torch", torch.__version__)

# --- device: cuda -> mps (Apple Silicon) -> cpu ---
if torch.cuda.is_available():
    DEVICE, dev_name = 0, "cuda:0"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE, dev_name = torch.device("mps"), "mps"
else:
    DEVICE, dev_name = -1, "cpu"
print("device:", dev_name)

# --- locate the faceted parquet regardless of where the notebook is launched ---
CANDIDATES = [
    Path("data/scaleup/queries/scaleup_queries_v2_faceted.parquet"),
    Path("../data/scaleup/queries/scaleup_queries_v2_faceted.parquet"),
    Path("scaleup_queries_v2_faceted.parquet"),
]
INPUT_PATH = next((p for p in CANDIDATES if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find scaleup_queries_v2_faceted.parquet; set INPUT_PATH manually.")
INPUT_PATH = INPUT_PATH.resolve()
DATA_DIR = INPUT_PATH.parent
print("input:", INPUT_PATH)

df = pd.read_parquet(INPUT_PATH)
print("shape:", df.shape)
df[["query_id", "query_text", "complexity", "technicality", "sensitivity"]].head()

transformers 5.9.0 | torch 2.12.0
device: mps
input: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/scaleup_queries_v2_faceted.parquet
shape: (250, 21)


,query_id,query_text,complexity,technicality,sensitivity
0,gb_1a77ad01ab570671,when does elena turn into a vampire in the tv ...,simple,non-technical,non-sensitive
1,gb_314f7fd2ce040788,who played the oldest brother in 7th heaven,simple,non-technical,non-sensitive
2,gb_eaab9147ac75176b,which came first the walking dead comic or show,simple,non-technical,non-sensitive
3,gb_51f95e9b9f4f6183,who plays unis in she's the man,simple,non-technical,NaN
4,gb_f3a09c028168dc20,where does the sound come from when you crack ...,simple,non-technical,NaN


## 2. Load the zero-shot NLI classifier

Loads the project NLI checkpoint as a `zero-shot-classification` pipeline. The first run
downloads ~1.6 GB; later runs use the local HF cache.

In [2]:
from transformers import pipeline

# Same checkpoint as thesis_config.NLI_MODEL (thesis_config.py:230) -> methodological consistency.
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
clf = pipeline("zero-shot-classification", model=NLI_MODEL, device=DEVICE)
print("loaded:", NLI_MODEL)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

loaded: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


## 3. Classification configuration

`hypothesis_template="{}"` means each *candidate label* is used verbatim as the NLI
hypothesis, so the exact sentences below are what the model scores against the query.
`label_map` converts the winning sentence back to the canonical GEO-Bench label that
already appears in the column.

In [3]:
FACET_CONFIG = {
    "complexity": {
        "hypothesis_template": "{}",
        "candidate_labels": [
            "This is a simple factual question.",
            "This is an intermediate question requiring some explanation.",
            "This is a complex question requiring detailed analysis.",
        ],
        "label_map": {
            "This is a simple factual question.": "simple",
            "This is an intermediate question requiring some explanation.": "intermediate",
            "This is a complex question requiring detailed analysis.": "complex",
        },
    },
    "technicality": {
        "hypothesis_template": "{}",
        "candidate_labels": [
            "This is a non-technical everyday question.",
            "This is a technical specialist question.",
        ],
        "label_map": {
            "This is a non-technical everyday question.": "non-technical",
            "This is a technical specialist question.": "technical",
        },
    },
    "sensitivity": {
        "hypothesis_template": "{}",
        "candidate_labels": [
            "This is a non-sensitive general knowledge question.",
            "This is a sensitive question involving health, finance, legal, or personal topics.",
        ],
        "label_map": {
            "This is a non-sensitive general knowledge question.": "non-sensitive",
            "This is a sensitive question involving health, finance, legal, or personal topics.": "sensitive",
        },
    },
}
FACETS = list(FACET_CONFIG)

# Sanity: candidate labels and label_map agree, and canonical values match what's in the data.
for col, c in FACET_CONFIG.items():
    assert set(c["candidate_labels"]) == set(c["label_map"]), col
    in_data = set(df[col].dropna().unique())
    canonical = set(c["label_map"].values())
    extra = in_data - canonical
    print(f"{col:13s} canonical={sorted(canonical)}  (unexpected in data: {sorted(extra) or 'none'})")

complexity    canonical=['complex', 'intermediate', 'simple']  (unexpected in data: none)
technicality  canonical=['non-technical', 'technical']  (unexpected in data: none)
sensitivity   canonical=['non-sensitive', 'sensitive']  (unexpected in data: none)


## 4. Identify missing values

A value is missing if it is `None`/`NaN` or an empty / whitespace-only string.

In [4]:
def is_missing(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return isinstance(v, str) and v.strip() == ""

# Capture the missing-row indices BEFORE filling (the consistency check relies on this).
missing_idx = {col: df.index[df[col].apply(is_missing)].tolist() for col in FACETS}
for col in FACETS:
    print(f"{col:13s} missing: {len(missing_idx[col]):3d} / {len(df)}")
total_to_fill = sum(len(v) for v in missing_idx.values())
print("TOTAL cells to fill:", total_to_fill)

complexity    missing:  31 / 250
technicality  missing:  29 / 250
sensitivity   missing:  51 / 250
TOTAL cells to fill: 111


## 5. Fill missing values (blanks only)

Only the rows captured above are classified; existing labels are asserted untouched.

In [5]:
def classify(query, col):
    c = FACET_CONFIG[col]
    res = clf(str(query), candidate_labels=c["candidate_labels"],
              hypothesis_template=c["hypothesis_template"])
    return c["label_map"][res["labels"][0]], float(res["scores"][0])

impute_log = []
for col in FACETS:
    for i in tqdm(missing_idx[col], desc=f"fill {col}"):
        assert is_missing(df.at[i, col]), "refusing to overwrite an existing label"
        label, score = classify(df.at[i, "query_text"], col)
        df.at[i, col] = label
        impute_log.append({
            "query_id": df.at[i, "query_id"],
            "query_text": df.at[i, "query_text"],
            "column_name": col,
            "imputed_value": label,
            "confidence_score": round(score, 4),
        })

print("filled:", len(impute_log), "cells")
pd.DataFrame(impute_log).head(10)

fill complexity:   0%|          | 0/31 [00:00<?, ?it/s]

fill technicality:   0%|          | 0/29 [00:00<?, ?it/s]

fill sensitivity:   0%|          | 0/51 [00:00<?, ?it/s]

filled: 111 cells


,query_id,query_text,column_name,imputed_value,confidence_score
0,gb_c30e3946381108e2,'Businesses owned by responsible and organized...,complexity,intermediate,0.7804
1,gb_1a90b9dd7be2bf64,login into onedrive,complexity,intermediate,0.6343
2,gb_8b26c2d8d5387ac3,wakehealth.edu,complexity,complex,0.4636
3,gb_157a7eb6b5972399,my service canada account log in,complexity,intermediate,0.4775
4,gb_e78889de403c8eaa,http://www.rewardclub.me/,complexity,complex,0.4794
5,gb_5b95c1501647091c,account/microsoft/login,complexity,intermediate,0.4573
6,gb_4ee9cc1110279ca5,prelicensetraining login,complexity,complex,0.4795
7,gb_c1ee9aa76d2cfa68,msnbc live streaming free online tv,complexity,intermediate,0.4757
8,gb_3dfd19034e427687,free online auto repair estimates,complexity,intermediate,0.5710
9,gb_780045e98ad742c0,grade to percentage calculator,complexity,intermediate,0.5053


## 6. Consistency check vs. existing labels (validation only)

Re-classify the rows that **already had** a label and measure agreement with the original
GEO-Bench labels. Nothing is overwritten here — this only produces the agreement statistic
and a sample of disagreements.

In [6]:
agreement_rows, disagreements = [], []
for col in FACETS:
    just_imputed = set(missing_idx[col])
    original = [i for i in df.index if i not in just_imputed and not is_missing(df.at[i, col])]
    agree = 0
    for i in tqdm(original, desc=f"check {col}"):
        pred, score = classify(df.at[i, "query_text"], col)
        if pred == df.at[i, col]:
            agree += 1
        else:
            disagreements.append({
                "column_name": col,
                "query_text": df.at[i, "query_text"],
                "existing": df.at[i, col],
                "predicted": pred,
                "confidence_score": round(score, 4),
            })
    n = len(original)
    pct = 100 * agree / n if n else float("nan")
    agreement_rows.append({"column": col, "checked": n, "agree": agree, "agreement_pct": round(pct, 1)})
    print(f"{col:13s} agreement: {agree}/{n} = {pct:.1f}%")

print("\n=== AGREEMENT SUMMARY ===")
display(pd.DataFrame(agreement_rows))
print("=== SAMPLE DISAGREEMENTS (max 10) ===")
display(pd.DataFrame(disagreements).head(10))

check complexity:   0%|          | 0/219 [00:00<?, ?it/s]

complexity    agreement: 98/219 = 44.7%


check technicality:   0%|          | 0/221 [00:00<?, ?it/s]

technicality  agreement: 91/221 = 41.2%


check sensitivity:   0%|          | 0/199 [00:00<?, ?it/s]

sensitivity   agreement: 107/199 = 53.8%

=== AGREEMENT SUMMARY ===


,column,checked,agree,agreement_pct
0,complexity,219,98,44.7
1,technicality,221,91,41.2
2,sensitivity,199,107,53.8


=== SAMPLE DISAGREEMENTS (max 10) ===


,column_name,query_text,existing,predicted,confidence_score
0,complexity,when does elena turn into a vampire in the tv ...,simple,intermediate,0.8830
1,complexity,who played the oldest brother in 7th heaven,simple,intermediate,0.7379
2,complexity,which came first the walking dead comic or show,simple,intermediate,0.8037
3,complexity,who plays unis in she's the man,simple,intermediate,0.6783
4,complexity,where does the sound come from when you crack ...,simple,intermediate,0.5259
5,complexity,the book of the thousand nights and one night ...,simple,complex,0.4943
6,complexity,who sings the christmas song all i want for ch...,simple,intermediate,0.4807
7,complexity,when did gaurdians of the galaxy 2 come out,simple,intermediate,0.7803
8,complexity,when will the next episode of flash be aired,simple,intermediate,0.6154
9,complexity,only fools and horses del falls through the ba...,simple,intermediate,0.5146


## 7. Summary and save

In [7]:
# Value counts after filling
for col in FACETS:
    print("---", col, "---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

# Confirm nothing remains missing in the three target columns
remaining = {col: int(df[col].apply(is_missing).sum()) for col in FACETS}
print("remaining missing:", remaining)
assert all(v == 0 for v in remaining.values()), "still missing values!"

OUT_PARQUET = DATA_DIR / "scaleup_queries_v2_faceted_filled.parquet"
OUT_LOG = DATA_DIR / "facet_imputation_log.csv"
df.to_parquet(OUT_PARQUET, index=False)
pd.DataFrame(impute_log).to_csv(OUT_LOG, index=False)
print("saved:", OUT_PARQUET)
print("saved:", OUT_LOG, f"({len(impute_log)} rows)")

--- complexity ---
complexity
intermediate    118
simple          104
complex          28

--- technicality ---
technicality
non-technical    189
technical         61

--- sensitivity ---
sensitivity
non-sensitive    166
sensitive         84

remaining missing: {'complexity': 0, 'technicality': 0, 'sensitivity': 0}


saved: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/scaleup_queries_v2_faceted_filled.parquet
saved: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/scaleup/queries/facet_imputation_log.csv (111 rows)
